# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [3]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [4]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_13836\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


In [ ]:
# ==========================================
# AR(1) — Pseudo-OOS continu (h=12) + Conformal PI (distribution)
# + Sauvegardes (bundle CI + modèle + méta)
# ==========================================

import os
import pickle
import joblib
import numpy as np
import pandas as pd

from dateutil.relativedelta import relativedelta
from statsmodels.tsa.ar_model import AutoReg

# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# "comme" PredictionIntervals / ConformalIntervals
STEP_SIZE = 12
PI_WINDOWS = 3
LEVEL = 95
alpha = 1 - (LEVEL / 100)

# Taille de calibration façon "n_windows=PI_WINDOWS" avec step_size=12
calib_size = PI_WINDOWS * STEP_SIZE  # 36 erreurs récentes
min_calib = calib_size               # (tu peux baisser à 24 si tu veux)

# ---------- Sécurisation de la série y ----------
y = (
    pd.Series(y.astype(float).values, index=pd.to_datetime(y.index))
      .asfreq("MS")
      .dropna()
)
print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

# ---------- Boucle pseudo-OOS continue ----------
rows = []
last_model = None
last_fit_end = None

# erreurs signées OOS (y_true - y_pred) pour "conformal_distribution"
past_signed_errors = []

last_t_end = y.index.max() - relativedelta(months=h)

for t_end in y.index:
    if t_end > last_t_end:
        break

    y_tr = y.loc[:t_end]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # Fit AR(1)
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # Point forecast à h
    fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    yhat_h = float(fc.iloc[-1])

    # Conformal interval (distribution)
    if len(past_signed_errors) >= min_calib:
        errs = np.array(past_signed_errors[-calib_size:])
        err_lo = float(np.quantile(errs, alpha / 2))
        err_hi = float(np.quantile(errs, 1 - alpha / 2))
        yhat_lo_95 = yhat_h + err_lo
        yhat_hi_95 = yhat_h + err_hi
    else:
        yhat_lo_95 = np.nan
        yhat_hi_95 = np.nan

    # Date cible et enregistrement si vérité dispo
    t_fore = t_end + relativedelta(months=h)
    if t_fore in y.index:
        y_true = float(y.loc[t_fore])

        # update calibration conformal
        past_signed_errors.append(y_true - yhat_h)

        rows.append((t_fore, yhat_h, y_true, yhat_lo_95, yhat_hi_95))

# ---------- DataFrame OOS ----------
if rows:
    df_oos_ar1 = (
        pd.DataFrame(rows, columns=["date", "y_hat", "y_true", "y_hat_lo_95", "y_hat_hi_95"])
          .set_index("date")
          .sort_index()
    )
else:
    df_oos_ar1 = pd.DataFrame(columns=["y_hat", "y_true", "y_hat_lo_95", "y_hat_hi_95"])
    df_oos_ar1.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_ar1)}")
print(df_oos_ar1.head(5))

# (option) check : première date avec CI dispo
first_ci = df_oos_ar1[df_oos_ar1["y_hat_lo_95"].notna()].head(1)
if len(first_ci):
    print("\n✅ Première prévision avec CI dispo :", first_ci.index[0].date())
else:
    print("\n⚠️ Pas encore de CI (min_calib trop élevé ou trop peu de données).")

# ============================================================
# Sauvegardes (noms avec suffixe CI)
# ============================================================
AR1_LAST_PKL  = "AR1_CI_last_trained_model.pkl"
AR1_LAST_META = "AR1_CI_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_CI_h12_oos_bundle.pkl"

# ============================================================
# 1) Sauvegarde du modèle final AR(1)
# ============================================================
if last_model is not None:
    try:
        joblib.dump(last_model, AR1_LAST_PKL)
        print(f"💾 Modèle AR(1) sauvegardé → {AR1_LAST_PKL}")
    except Exception:
        with open(AR1_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(1) sauvegardé (pickle) → {AR1_LAST_PKL}")

# ============================================================
# 2) Bundle des sorties OOS (AVEC CI conformal)
# ============================================================
bundle = {
    "oos_predictions": (
        df_oos_ar1
        .reset_index()
        .rename(columns={
            "y_hat": "y_pred",
            "y_hat_lo_95": "y_pred_lo_95",
            "y_hat_hi_95": "y_pred_hi_95",
        })
        .assign(
            date=lambda d: (
                pd.to_datetime(d["date"])
                  .dt.to_period("M")
                  .dt.to_timestamp(how="start")
            )
        )
    ),
    "params": {
        "model": "AR(1)",
        "trend": trend,
        "horizon": h,
        "lag": 1,
        "min_train_n": min_train_n,
        "ci_method": "conformal_distribution",
        "ci_level": int(LEVEL),
        "pi_windows": int(PI_WINDOWS),
        "step_size": int(STEP_SIZE),
    },
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_ar1)),
    }
}

with open(AR1_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)

print(f"💾 Bundle AR(1) OOS (CI) sauvegardé → {AR1_BUNDLE}")

# ============================================================
# 3) Méta CSV (clean, CI)
# ============================================================
meta_row = {
    "model": "AR(1)",
    "trend": trend,
    "lag": 1,
    "horizon": h,
    "ci_method": "conformal_distribution",
    "ci_level": int(LEVEL),
    "pi_windows": int(PI_WINDOWS),
    "step_size": int(STEP_SIZE),
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_ar1)),
}

pd.DataFrame([meta_row]).to_csv(AR1_LAST_META, index=False)
print(f"💾 Méta AR(1) (CI) sauvegardée → {AR1_LAST_META}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS

✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  y_hat_lo_95  y_hat_hi_95
date                                                  
1963-12-01 -0.080890     0.0          NaN          NaN
1964-01-01  0.141077    -0.1          NaN          NaN
1964-02-01  0.408114    -0.5          NaN          NaN
1964-03-01  0.242637    -0.3          NaN          NaN
1964-04-01  0.238955    -0.4          NaN          NaN

✅ Première prévision avec CI dispo : 1966-12-01
💾 Modèle AR(1) sauvegardé → AR1_CI_last_trained_model.pkl
💾 Bundle AR(1) OOS (CI) sauvegardé → AR1_CI_h12_oos_bundle.pkl
💾 Méta AR(1) (CI) sauvegardée → AR1_CI_last_trained_model_meta.csv
